# D3 Stitching Evaluation for cam03

Evaluate proposed vs selected edges with full D2 cost breakdown, and compare against manual entity mapping (with optional aggregatable partner pairs).

## 1. Setup and Paths

Configure project paths and the cam03 clip ID we are evaluating.

In [55]:
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

PROJECT_ROOT = Path("/Users/bryanthomas/Desktop/Professional/Projects/roll_tracker")
CLIP_ID = "cam03-20260103-124000_0-30s"
CAMERA_ID = "cam03"

OUTPUT_ROOT = PROJECT_ROOT / "outputs" / CLIP_ID
STAGE_D_DIR = OUTPUT_ROOT / "stage_D"
DEBUG_DIR = OUTPUT_ROOT / "_debug"

# Core artifacts
D1_NODES_PATH = STAGE_D_DIR / "d1_graph_nodes.parquet"
D1_EDGES_PATH = STAGE_D_DIR / "d1_graph_edges.parquet"
D2_COSTS_PATH = STAGE_D_DIR / "d2_edge_costs.parquet"
D3_COMPILED_EDGES_PATH = DEBUG_DIR / "d3_compiled_edges_pruned.parquet"
D3_COMPILED_COSTS_PATH = DEBUG_DIR / "d3_compiled_costs_pruned.parquet"
D3_SELECTED_EDGES_PATH = DEBUG_DIR / "d3_selected_edges.parquet"
TRACKLET_FRAMES_PATH = STAGE_D_DIR / "tracklet_bank_frames.parquet"
TRACKLET_SUMMARIES_PATH = STAGE_D_DIR / "tracklet_bank_summaries.parquet"
D3_ENTITIES_JSON_PATH = DEBUG_DIR / "d3_entities_format_a.json"

print("Stage D dir:", STAGE_D_DIR)
print("D2 costs exists:", D2_COSTS_PATH.exists())
print("D3 compiled costs exists:", D3_COMPILED_COSTS_PATH.exists())
print("D3 selected edges exists:", D3_SELECTED_EDGES_PATH.exists())
print("Tracklet frames exists:", TRACKLET_FRAMES_PATH.exists())

Stage D dir: /Users/bryanthomas/Desktop/Professional/Projects/roll_tracker/outputs/cam03-20260103-124000_0-30s/stage_D
D2 costs exists: True
D3 compiled costs exists: True
D3 selected edges exists: True
Tracklet frames exists: True


## 2. Load D1/D2/D3 Edge Tables

Here we load D1 nodes/edges, D2 compiled costs, and D3 selected edges into pandas DataFrames.

In [56]:
# Load D1 nodes and edges
d1_nodes = pq.read_table(D1_NODES_PATH).to_pandas()
d1_edges = pq.read_table(D1_EDGES_PATH).to_pandas()

print("D1 nodes:", d1_nodes.shape, "columns:", sorted(d1_nodes.columns.tolist()))
print("D1 edges:", d1_edges.shape, "columns:", sorted(d1_edges.columns.tolist()))

# Load D3 compiled (pruned) costs and edges
d3_compiled_costs = pq.read_table(D3_COMPILED_COSTS_PATH).to_pandas()
d3_compiled_edges = pq.read_table(D3_COMPILED_EDGES_PATH).to_pandas()

print("D3 compiled costs:", d3_compiled_costs.shape)
print("D3 compiled edges:", d3_compiled_edges.shape)

# Load D3 selected edges
d3_selected = pq.read_table(D3_SELECTED_EDGES_PATH).to_pandas()
print("D3 selected edges:", d3_selected.shape)

d3_compiled_costs.head()

D1 nodes: (33, 14) columns: ['base_tracklet_id', 'cannot_link_anchor_keys_json', 'capacity', 'carrier_tracklet_id', 'disappearing_tracklet_id', 'end_frame', 'must_link_anchor_key', 'must_link_confidence', 'new_tracklet_id', 'node_id', 'node_type', 'payload_json', 'segment_type', 'start_frame']
D1 edges: (72, 9) columns: ['capacity', 'dt_frames', 'edge_id', 'edge_type', 'merge_end', 'payload_json', 'split_start', 'u', 'v']
D3 compiled costs: (72, 24)
D3 compiled edges: (72, 9)
D3 selected edges: (35, 11)


,edge_id,edge_type,src_node_id,dst_node_id,is_allowed,disallow_reasons_json,dt_frames,dt_s,dist_m,v_req_mps,...,term_time,term_vreq,term_missing_geom,term_flags,term_group_coherence,term_birth_prior,term_death_prior,term_merge_prior,term_split_prior,total_cost
0,E:BIRTH:G:0-137:carrier=t1:d=none:n=t4,EdgeType.BIRTH,SOURCE,G:0-137:carrier=t1:d=none:n=t4,True,[],<NA>,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,2.01
1,E:BIRTH:G:11-237:carrier=t3:d=none:n=t7,EdgeType.BIRTH,SOURCE,G:11-237:carrier=t3:d=none:n=t7,True,[],<NA>,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,2.01
2,E:BIRTH:G:174-217:carrier=t5:d=none:n=t6,EdgeType.BIRTH,SOURCE,G:174-217:carrier=t5:d=none:n=t6,True,[],<NA>,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,2.01
3,E:BIRTH:G:805-852:carrier=t15:d=none:n=t16,EdgeType.BIRTH,SOURCE,G:805-852:carrier=t15:d=none:n=t16,True,[],<NA>,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,2.01
4,E:BIRTH:T:t10,EdgeType.BIRTH,SOURCE,T:t10,True,[],<NA>,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,2.01


## 3. Build Unified Edge Table with Costs and Selection Flags

Join compiled edges with compiled costs, then mark which edges were selected by D3 via a boolean flag.

In [57]:
# Mark selected edges and join compiled edges with compiled costs

# Selected edge_ids from D3
d3_selected_ids = set(d3_selected["edge_id"].astype(str))

# Ensure edge_id columns are strings
d3_compiled_edges["edge_id"] = d3_compiled_edges["edge_id"].astype(str)
d3_compiled_costs["edge_id"] = d3_compiled_costs["edge_id"].astype(str)

# Join edges with costs on edge_id
edges_with_costs = d3_compiled_edges.merge(d3_compiled_costs, on="edge_id", how="left", suffixes=("_edge", "_cost"))

# Add selection flag and flow (if present in d3_selected)
sel = d3_selected[["edge_id", "flow"]].copy()
sel["edge_id"] = sel["edge_id"].astype(str)

edges_with_costs = edges_with_costs.merge(sel, on="edge_id", how="left")
edges_with_costs["is_selected"] = edges_with_costs["flow"].fillna(0).astype(int) > 0

print("Unified edge table shape:", edges_with_costs.shape)
print("Columns (first 20):", sorted(edges_with_costs.columns.tolist())[:20])

edges_with_costs.head()

Unified edge table shape: (72, 34)
Columns (first 20): ['capacity', 'contact_rel', 'disallow_reasons_json', 'dist_m', 'dist_norm', 'dst_node_id', 'dt_frames_cost', 'dt_frames_edge', 'dt_s', 'edge_id', 'edge_type_cost', 'edge_type_edge', 'endpoint_flagged', 'flow', 'is_allowed', 'is_selected', 'merge_end', 'payload_json', 'split_start', 'src_node_id']


,edge_id,edge_type_edge,u,v,capacity,dt_frames_edge,merge_end,split_start,payload_json,edge_type_cost,...,term_missing_geom,term_flags,term_group_coherence,term_birth_prior,term_death_prior,term_merge_prior,term_split_prior,total_cost,flow,is_selected
0,E:BIRTH:G:0-137:carrier=t1:d=none:n=t4,EdgeType.BIRTH,SOURCE,G:0-137:carrier=t1:d=none:n=t4,2,<NA>,<NA>,<NA>,{},EdgeType.BIRTH,...,0.0,0.0,0.0,2.0,0.0,0.0,0.0,2.01,2.0,True
1,E:BIRTH:G:11-237:carrier=t3:d=none:n=t7,EdgeType.BIRTH,SOURCE,G:11-237:carrier=t3:d=none:n=t7,2,<NA>,<NA>,<NA>,{},EdgeType.BIRTH,...,0.0,0.0,0.0,2.0,0.0,0.0,0.0,2.01,NaN,False
2,E:BIRTH:G:174-217:carrier=t5:d=none:n=t6,EdgeType.BIRTH,SOURCE,G:174-217:carrier=t5:d=none:n=t6,2,<NA>,<NA>,<NA>,{},EdgeType.BIRTH,...,0.0,0.0,0.0,2.0,0.0,0.0,0.0,2.01,2.0,True
3,E:BIRTH:G:805-852:carrier=t15:d=none:n=t16,EdgeType.BIRTH,SOURCE,G:805-852:carrier=t15:d=none:n=t16,2,<NA>,<NA>,<NA>,{},EdgeType.BIRTH,...,0.0,0.0,0.0,2.0,0.0,0.0,0.0,2.01,NaN,False
4,E:BIRTH:T:t10,EdgeType.BIRTH,SOURCE,T:t10,1,<NA>,<NA>,<NA>,{},EdgeType.BIRTH,...,0.0,0.0,0.0,2.0,0.0,0.0,0.0,2.01,NaN,False


## 4. Attach Base Tracklet IDs to Edges

Join D1 node metadata so each edge has `base_tracklet_id` for its source and destination nodes.

In [58]:
# Index D1 nodes by node_id for joins
nodes_idx = d1_nodes.set_index("node_id")

# Helper to map node_id -> base_tracklet_id and segment_type
u_meta = nodes_idx[["base_tracklet_id", "segment_type"]].rename(
    columns={"base_tracklet_id": "u_base_tracklet_id", "segment_type": "u_segment_type"}
)
v_meta = nodes_idx[["base_tracklet_id", "segment_type"]].rename(
    columns={"base_tracklet_id": "v_base_tracklet_id", "segment_type": "v_segment_type"}
)

edges_enriched = edges_with_costs.merge(
    u_meta, left_on="u", right_index=True, how="left"
).merge(
    v_meta, left_on="v", right_index=True, how="left"
)

print("Enriched edge table shape:", edges_enriched.shape)
print("Sample columns:", [c for c in edges_enriched.columns if "base_tracklet_id" in c or "segment_type" in c])

edges_enriched.head()

Enriched edge table shape: (72, 38)
Sample columns: ['u_base_tracklet_id', 'u_segment_type', 'v_base_tracklet_id', 'v_segment_type']


,edge_id,edge_type_edge,u,v,capacity,dt_frames_edge,merge_end,split_start,payload_json,edge_type_cost,...,term_death_prior,term_merge_prior,term_split_prior,total_cost,flow,is_selected,u_base_tracklet_id,u_segment_type,v_base_tracklet_id,v_segment_type
0,E:BIRTH:G:0-137:carrier=t1:d=none:n=t4,EdgeType.BIRTH,SOURCE,G:0-137:carrier=t1:d=none:n=t4,2,<NA>,<NA>,<NA>,{},EdgeType.BIRTH,...,0.0,0.0,0.0,2.01,2.0,True,None,None,t1,GROUP
1,E:BIRTH:G:11-237:carrier=t3:d=none:n=t7,EdgeType.BIRTH,SOURCE,G:11-237:carrier=t3:d=none:n=t7,2,<NA>,<NA>,<NA>,{},EdgeType.BIRTH,...,0.0,0.0,0.0,2.01,NaN,False,None,None,t3,GROUP
2,E:BIRTH:G:174-217:carrier=t5:d=none:n=t6,EdgeType.BIRTH,SOURCE,G:174-217:carrier=t5:d=none:n=t6,2,<NA>,<NA>,<NA>,{},EdgeType.BIRTH,...,0.0,0.0,0.0,2.01,2.0,True,None,None,t5,GROUP
3,E:BIRTH:G:805-852:carrier=t15:d=none:n=t16,EdgeType.BIRTH,SOURCE,G:805-852:carrier=t15:d=none:n=t16,2,<NA>,<NA>,<NA>,{},EdgeType.BIRTH,...,0.0,0.0,0.0,2.01,NaN,False,None,None,t15,GROUP
4,E:BIRTH:T:t10,EdgeType.BIRTH,SOURCE,T:t10,1,<NA>,<NA>,<NA>,{},EdgeType.BIRTH,...,0.0,0.0,0.0,2.01,NaN,False,None,None,t10,SOLO


## 5. Hook for Manual GT Mapping (from temp.txt)

Parse a simple mapping from tracklet IDs to ground-truth entities based on the current `roll_tracker/temp.txt` format (best-effort regex).

In [59]:
# Attach GT labels to edges and classify stitches
import pandas as pd

edges = edges_enriched.copy()

def lookup_gt(tid: str) -> str | None:
    return tracklet_to_gt.get(tid)

edges["u_gt"] = edges["u_base_tracklet_id"].map(lookup_gt)
edges["v_gt"] = edges["v_base_tracklet_id"].map(lookup_gt)

def classify_edge(row: pd.Series) -> str:
    u_gt = row["u_gt"]
    v_gt = row["v_gt"]

    # If either endpoint lacks GT, treat as 'unknown' for now.
    if pd.isna(u_gt) or pd.isna(v_gt):
        return "unknown"

    same_ent = u_gt == v_gt

    # For now, we regard any edge connecting different GT entities as 'bad'.
    return "good" if same_ent else "bad"

edges["gt_class"] = edges.apply(classify_edge, axis=1)

# Focus on selected edges only for an initial view.
sel = edges[edges["is_selected"]].copy()
print("Selected edges by GT class and edge_type_cost:")
print(sel.groupby(["gt_class", "edge_type_cost"]).size())

# Look specifically at 'bad' selected edges and their main cost terms.
bad_sel = sel[sel["gt_class"] == "bad"].copy()
cols_of_interest = ["u_base_tracklet_id","v_base_tracklet_id","edge_type_cost","total_cost","term_env","term_time","term_vreq","term_group_coherence","term_flags","term_missing_geom"]
print("\nBad selected edges (cross-GT):")
display(bad_sel[cols_of_interest].sort_values("total_cost"))

# Keep this DataFrame around for further ad-hoc analysis.
edges_with_gt = edges

Selected edges by GT class and edge_type_cost:
gt_class  edge_type_cost   
bad       EdgeType.CONTINUE     1
          EdgeType.MERGE        3
          EdgeType.SPLIT        3
good      EdgeType.CONTINUE    12
          EdgeType.MERGE        2
          EdgeType.SPLIT        3
unknown   EdgeType.BIRTH        5
          EdgeType.DEATH        6
dtype: int64

Bad selected edges (cross-GT):


,u_base_tracklet_id,v_base_tracklet_id,edge_type_cost,total_cost,term_env,term_time,term_vreq,term_group_coherence,term_flags,term_missing_geom
56,t10,t4,EdgeType.MERGE,-0.390000,0.01,0.0,0.000000,-0.5,0.0,0.0
68,t4,t10,EdgeType.SPLIT,-0.388454,0.01,0.0,0.001546,-0.5,0.0,0.0
67,t4,t9,EdgeType.SPLIT,-0.383390,0.01,0.0,0.006610,-0.5,0.0,0.0
69,t4,t13,EdgeType.SPLIT,-0.377808,0.01,0.0,0.012192,-0.5,0.0,0.0
62,t9,t4,EdgeType.MERGE,-0.365252,0.01,0.0,0.024748,-0.5,0.0,0.0
57,t13,t4,EdgeType.MERGE,-0.097898,0.01,0.0,0.292102,-0.5,0.0,0.0
38,t7,t8,EdgeType.CONTINUE,1.010000,1.01,0.0,0.000000,0.0,0.0,0.0


In [60]:
import re

temp_path = PROJECT_ROOT / "temp.txt"
print("Manual GT temp file:", temp_path, "exists:", temp_path.exists())

entity_to_tids = {}
if temp_path.exists():
    text = temp_path.read_text()
    # The current temp.txt is a loose JSON-like structure, e.g.:
    #   "entity_1": {"t1", ...},
    #   {"april_tag_found_in": "none"};
    # We parse it line-by-line, looking only at the entity_* lines and
    # extracting all tokens that look like tracklet IDs (t1, t2, ...).
    line_pattern = re.compile(r'"entity_(\d+)"\s*:\s*\{([^}]*)\}', flags=re.IGNORECASE)
    tid_pattern = re.compile(r"t\d+")
    for line in text.splitlines():
        m = line_pattern.search(line)
        if not m:
            continue
        ent_num = m.group(1)
        block = m.group(2)
        tids = sorted(set(tid_pattern.findall(block)))
        if tids:
            entity_to_tids[f"E{ent_num}"] = tids

print("Parsed GT entities (best-effort):", entity_to_tids)

# Build a reverse map: tracklet_id -> gt_entity_id (first one if ambiguous)
tracklet_to_gt = {}
for ent_id, tids in entity_to_tids.items():
    for tid in tids:
        tracklet_to_gt.setdefault(tid, ent_id)

print("Example GT assignments:", list(tracklet_to_gt.items())[:10])

Manual GT temp file: /Users/bryanthomas/Desktop/Professional/Projects/roll_tracker/temp.txt exists: True
Parsed GT entities (best-effort): {'E1': ['t1', 't4'], 'E2': ['t1', 't10', 't13', 't4', 't9'], 'E3': ['t16', 't5', 't6'], 'E4': ['t6'], 'E5': ['t15', 't2'], 'E6': ['t11', 't14', 't3', 't8'], 'E7': ['t11', 't14', 't3', 't7', 't8']}
Example GT assignments: [('t1', 'E1'), ('t4', 'E1'), ('t10', 'E2'), ('t13', 'E2'), ('t9', 'E2'), ('t16', 'E3'), ('t5', 'E3'), ('t6', 'E3'), ('t15', 'E5'), ('t2', 'E5')]


In [61]:
# Aggregate cost terms for good vs bad selected edges
term_cols = [c for c in edges_with_gt.columns if c.startswith("term_")]
gb = edges_with_gt[edges_with_gt["is_selected"] & edges_with_gt["gt_class"].isin(["good", "bad"])].copy()
summary = gb.groupby("gt_class")[term_cols + ["total_cost"]].mean().T
print("Mean costs by gt_class (rows = terms, cols = good/bad):")
display(summary)

Mean costs by gt_class (rows = terms, cols = good/bad):


gt_class,bad,good
term_env,0.152857,0.068824
term_time,0.000000,0.002117
term_vreq,0.048171,0.201308
term_missing_geom,0.000000,0.000000
term_flags,0.000000,0.102941
term_group_coherence,-0.428571,-0.147059
term_birth_prior,0.000000,0.000000
term_death_prior,0.000000,0.000000
term_merge_prior,0.042857,0.011765
term_split_prior,0.042857,0.017647


## 6. Inspect D2 Cost Config and Explain Individual Edges


Use the Stage D D2 cost config together with per-edge features to explain why specific good/bad edges receive their prices.

In [62]:
# Load Stage D D2 cost config (default + optional camera override)
import yaml
from pprint import pprint

default_cfg_path = PROJECT_ROOT / "configs" / "default.yaml"
camera_cfg_path = PROJECT_ROOT / "configs" / "cameras" / f"{CAMERA_ID}.yaml"

print("Default config path:", default_cfg_path)
print("Camera override path:", camera_cfg_path, "exists:", camera_cfg_path.exists())

with default_cfg_path.open("r") as f:
    default_cfg = yaml.safe_load(f)

d2_cfg = dict(default_cfg["stages"]["stage_D"]["d2_costs"])  # shallow copy

if camera_cfg_path.exists():
    with camera_cfg_path.open("r") as f:
        cam_cfg = yaml.safe_load(f) or {}
    cam_d2 = (
        cam_cfg.get("stages", {})
        .get("stage_D", {})
        .get("d2_costs", {})
    )
    if cam_d2:
        d2_cfg.update(cam_d2)

print("\nEffective D2 cost config (merged):")
pprint(d2_cfg)

Default config path: /Users/bryanthomas/Desktop/Professional/Projects/roll_tracker/configs/default.yaml
Camera override path: /Users/bryanthomas/Desktop/Professional/Projects/roll_tracker/configs/cameras/cam03.yaml exists: False

Effective D2 cost config (merged):
{'base_env_cost': 0.01,
 'birth_cost': 2.0,
 'bonus_group_coherent': 0.5,
 'contact_conf_floor': 0.25,
 'contact_rel_alpha': 0.35,
 'death_cost': 2.0,
 'dt_max_s': 1.0,
 'enabled': True,
 'endpoint_search_window_frames': None,
 'merge_prior': 0.1,
 'missing_geom_policy': 'disallow',
 'penalty_group_incoherent': 0.5,
 'reconnect_extra_env_cost': 1.0,
 'reconnect_w_time': 0.0,
 'reconnect_w_z': 1.0,
 'split_prior': 0.1,
 'use_contact_rel': True,
 'use_flags': True,
 'v_cost_scale_mps': None,
 'v_hinge_mps': None,
 'w_flags': 0.25,
 'w_time': 0.1,
 'w_vreq': 1.0}


In [63]:
# Helper to explain why a single edge has its cost, term by term
import math

def explain_edge(row, d2_cfg, max_flags_to_show=8):
    print("=" * 80)
    print(f"edge_id: {row.get('edge_id')}")
    print(f"u: {row.get('u')} (base_tracklet={row.get('u_base_tracklet_id')}, gt={row.get('u_gt')})")
    print(f"v: {row.get('v')} (base_tracklet={row.get('v_base_tracklet_id')}, gt={row.get('v_gt')})")
    print(f"edge_type: {row.get('edge_type_cost')}  |  gt_class: {row.get('gt_class')}  |  is_selected: {bool(row.get('is_selected'))}")
    print(f"total_cost: {row.get('total_cost')}")
    print()

    # Common scalar features (best-effort; some may be NaN depending on edge_type)
    dist_m = row.get("dist_m")
    dt_s = row.get("dt_s")
    v_req = row.get("v_req_mps")
    contact_rel = row.get("contact_rel")
    print("Features:")
    print(f"  dist_m: {dist_m}")
    print(f"  dt_s: {dt_s}")
    print(f"  v_req_mps: {v_req}")
    print(f"  contact_rel: {contact_rel}")
    print()

    # term_vreq
    term_vreq = row.get("term_vreq")
    w_vreq = d2_cfg.get("w_vreq")
    v_scale = d2_cfg.get("v_cost_scale_mps")
    v_hinge = d2_cfg.get("v_hinge_mps")
    print("term_vreq (motion + MERGE/SPLIT distance):")
    print(f"  value: {term_vreq}")
    print(f"  config: w_vreq={w_vreq}, v_cost_scale_mps={v_scale}, v_hinge_mps={v_hinge}")
    print("  intuition: higher required velocity / distance beyond thresholds should increase this term;")
    print("            currently bad edges often have lower term_vreq than good ones → vreq is relatively weak.")
    print()

    # term_group_coherence
    term_gc = row.get("term_group_coherence")
    bonus_gc = d2_cfg.get("bonus_group_coherent")
    penalty_gc = d2_cfg.get("penalty_group_incoherent")
    # Try to infer a simple coherence flag if present
    group_coherent = row.get("group_coherent", math.nan)
    print("term_group_coherence (group behavior):")
    print(f"  value: {term_gc}")
    print(f"  config: bonus_group_coherent={bonus_gc}, penalty_group_incoherent={penalty_gc}")
    print(f"  raw group_coherent feature (if present): {group_coherent}")
    print("  intuition: strongly negative values mean the solver is being *rewarded* for this edge on group grounds;")
    print("            our aggregates show bad edges enjoy more negative term_group_coherence than good ones.")
    print()

    # term_env
    term_env = row.get("term_env")
    base_env = d2_cfg.get("base_env_cost")
    reconnect_extra = d2_cfg.get("reconnect_extra_env_cost")
    print("term_env (environment / reconnect-specific costs):")
    print(f"  value: {term_env}")
    print(f"  config: base_env_cost={base_env}, reconnect_extra_env_cost={reconnect_extra}")
    print("  intuition: environment costs add a small baseline and extra cost for reconnects;")
    print("            in our aggregates, bad edges pay slightly more env cost, so this term is not the driver.")
    print()

    # term_flags
    term_flags = row.get("term_flags")
    use_flags = d2_cfg.get("use_flags")
    w_flags = d2_cfg.get("w_flags")
    # Heuristically show any boolean-like columns that look like flags
    flag_cols = [c for c in row.index if c.startswith("flag_") or c.endswith("_flagged")]
    active_flags = {c: row[c] for c in flag_cols if bool(row[c])}
    print("term_flags (penalties from diagnostic flags):")
    print(f"  value: {term_flags}")
    print(f"  config: use_flags={use_flags}, w_flags={w_flags}")
    if active_flags:
        print("  active flags (subset):")
        for i, (name, val) in enumerate(active_flags.items()):
            if i >= max_flags_to_show:
                print("    … (more flags omitted)")
                break
            print(f"    {name}: {val}")
    else:
        print("  active flags: none detected on this edge (all relevant flags false or absent)")
    print("  intuition: flags should penalize suspicious edges; our aggregates show flags often hit good edges more.")
    print()

    # Priors
    print("Priors (birth/death/merge/split):")
    for name in ["term_birth_prior", "term_death_prior", "term_merge_prior", "term_split_prior"]:
        print(f"  {name}: {row.get(name)}")
    print("  config priors: birth_cost={birth_cost}, death_cost={death_cost}, merge_prior={merge_prior}, split_prior={split_prior}".format(
          birth_cost=d2_cfg.get("birth_cost"),
          death_cost=d2_cfg.get("death_cost"),
          merge_prior=d2_cfg.get("merge_prior"),
          split_prior=d2_cfg.get("split_prior"),
) )
    print()

    print("Decomposition check (selected terms → total_cost):")
    terms = [
        row.get("term_env"),
        row.get("term_time"),
        row.get("term_vreq"),
        row.get("term_missing_geom"),
        row.get("term_flags"),
        row.get("term_group_coherence"),
        row.get("term_birth_prior"),
        row.get("term_death_prior"),
        row.get("term_merge_prior"),
        row.get("term_split_prior"),
    ]
    approx_sum = sum(t for t in terms if pd.notna(t))
    print(f"  sum(selected term_*) ≈ {approx_sum}")
    print(f"  reported total_cost = {row.get('total_cost')}")
    print("=" * 80)


def explain_sample_edges(edges_with_gt, d2_cfg, n_per_type=2):
    """Sample a few bad vs good edges and print detailed explanations."""
    sel = edges_with_gt[edges_with_gt["is_selected"]].copy()
    bad = sel[sel["gt_class"] == "bad"]
    good = sel[sel["gt_class"] == "good"]

    print("Bad selected edges (count =", len(bad), ") — sorted by total_cost:")
    display(bad.sort_values("total_cost")[[
        "edge_id","edge_type_cost","u_base_tracklet_id","v_base_tracklet_id","total_cost"]].head(10))
    print()
    print("Good selected edges (count =", len(good), ") — sorted by total_cost:")
    display(good.sort_values("total_cost")[[
        "edge_id","edge_type_cost","u_base_tracklet_id","v_base_tracklet_id","total_cost"]].head(10))
    print()

    # For a quick first pass, just pick the cheapest few bad and good edges overall
    print("\nDetailed explanations for a few BAD selected edges:")
    for _, row in bad.sort_values("total_cost").head(n_per_type).iterrows():
        explain_edge(row, d2_cfg)

    print("\nDetailed explanations for a few GOOD selected edges:")
    for _, row in good.sort_values("total_cost").head(n_per_type).iterrows():
        explain_edge(row, d2_cfg)

In [64]:
# Run explanations for a small sample of edges (adjust n_per_type as needed)
explain_sample_edges(edges_with_gt, d2_cfg, n_per_type=3)

Bad selected edges (count = 7 ) — sorted by total_cost:


,edge_id,edge_type_cost,u_base_tracklet_id,v_base_tracklet_id,total_cost
56,E:MERGE:T:t10->G:548-757:carrier=t4:d=t10:n=t13,EdgeType.MERGE,t10,t4,-0.390000
68,E:SPLIT:G:427-507:carrier=t4:d=t9:n=t10->T:t10,EdgeType.SPLIT,t4,t10,-0.388454
67,E:SPLIT:G:281-332:carrier=t4:d=t1:n=t9->T:t9,EdgeType.SPLIT,t4,t9,-0.383390
69,E:SPLIT:G:548-757:carrier=t4:d=t10:n=t13->T:t13,EdgeType.SPLIT,t4,t13,-0.377808
62,E:MERGE:T:t9->G:427-507:carrier=t4:d=t9:n=t10,EdgeType.MERGE,t9,t4,-0.365252
57,E:MERGE:T:t13->G:811-900:carrier=t4:d=t13:n=none,EdgeType.MERGE,t13,t4,-0.097898
38,E:CONT_RECONNECT:T:t7->T:t8,EdgeType.CONTINUE,t7,t8,1.010000



Good selected edges (count = 17 ) — sorted by total_cost:


,edge_id,edge_type_cost,u_base_tracklet_id,v_base_tracklet_id,total_cost
63,E:SPLIT:G:0-137:carrier=t1:d=none:n=t4->T:t4:s...,EdgeType.SPLIT,t1,t4,-0.250810
58,E:MERGE:T:t1:s1:138-280->G:281-332:carrier=t4:...,EdgeType.MERGE,t1,t4,-0.156189
65,E:SPLIT:G:174-217:carrier=t5:d=none:n=t6->T:t6...,EdgeType.SPLIT,t5,t6,-0.035753
15,E:CONT:G:0-137:carrier=t1:d=none:n=t4->T:t1:s1...,EdgeType.CONTINUE,t1,t1,0.013272
26,E:CONT:T:t4:s0:138-280->G:281-332:carrier=t4:d...,EdgeType.CONTINUE,t4,t4,0.013272
28,E:CONT:T:t4:s4:508-547->G:548-757:carrier=t4:d...,EdgeType.CONTINUE,t4,t4,0.013272
21,E:CONT:G:548-757:carrier=t4:d=t10:n=t13->T:t4:...,EdgeType.CONTINUE,t4,t4,0.013272
20,E:CONT:G:427-507:carrier=t4:d=t9:n=t10->T:t4:s...,EdgeType.CONTINUE,t4,t4,0.263272
29,E:CONT:T:t4:s6:758-810->G:811-900:carrier=t4:d...,EdgeType.CONTINUE,t4,t4,0.263272
19,E:CONT:G:281-332:carrier=t4:d=t1:n=t9->T:t4:s2...,EdgeType.CONTINUE,t4,t4,0.263272




Detailed explanations for a few BAD selected edges:
edge_id: E:MERGE:T:t10->G:548-757:carrier=t4:d=t10:n=t13
u: T:t10 (base_tracklet=t10, gt=E2)
v: G:548-757:carrier=t4:d=t10:n=t13 (base_tracklet=t4, gt=E1)
edge_type: EdgeType.MERGE  |  gt_class: bad  |  is_selected: True
total_cost: -0.39

Features:
  dist_m: 0.0
  dt_s: nan
  v_req_mps: nan
  contact_rel: nan

term_vreq (motion + MERGE/SPLIT distance):
  value: 0.0
  config: w_vreq=1.0, v_cost_scale_mps=None, v_hinge_mps=None
  intuition: higher required velocity / distance beyond thresholds should increase this term;
            currently bad edges often have lower term_vreq than good ones → vreq is relatively weak.

term_group_coherence (group behavior):
  value: -0.5
  config: bonus_group_coherent=0.5, penalty_group_incoherent=0.5
  raw group_coherent feature (if present): nan
  intuition: strongly negative values mean the solver is being *rewarded* for this edge on group grounds;
            our aggregates show bad edges enjoy 

## 7. Pairwise Comparison: Bad vs Best Good Alternative

For each selected bad edge (cross-GT), find the best-scoring selected good edge that touches the same base tracklet(s) and compare term-by-term deltas to see which costs drive the mis-ranking.

In [65]:
# For each bad selected edge, try to find a "best" good alternative
import pandas as pd

bad_sel = edges_with_gt[(edges_with_gt["is_selected"]) & (edges_with_gt["gt_class"] == "bad")].copy()
good_sel = edges_with_gt[(edges_with_gt["is_selected"]) & (edges_with_gt["gt_class"] == "good")].copy()

print("Bad selected edges:", len(bad_sel))
print("Good selected edges:", len(good_sel))

def find_best_alt_edge(row_bad: pd.Series, good_edges: pd.DataFrame) -> pd.Series | None:
    """Find the lowest-cost good edge that shares a base tracklet and edge_type_cost."""
    candidates = good_edges
    edge_type = row_bad.get("edge_type_cost")
    if edge_type is not None:
        candidates = candidates[candidates["edge_type_cost"] == edge_type]
    if candidates.empty:
        return None

    u_bt = row_bad.get("u_base_tracklet_id")
    v_bt = row_bad.get("v_base_tracklet_id")
    mask = pd.Series(False, index=candidates.index)
    if pd.notna(u_bt):
        mask |= (candidates["u_base_tracklet_id"] == u_bt) | (candidates["v_base_tracklet_id"] == u_bt)
    if pd.notna(v_bt):
        mask |= (candidates["u_base_tracklet_id"] == v_bt) | (candidates["v_base_tracklet_id"] == v_bt)
    candidates = candidates[mask]
    if candidates.empty:
        return None
    return candidates.sort_values("total_cost").iloc[0]

pair_rows = []
for _, rb in bad_sel.iterrows():
    alt = find_best_alt_edge(rb, good_sel)
    if alt is None:
        continue
    row = {
        "bad_edge_id": rb["edge_id"],
        "alt_edge_id": alt["edge_id"],
        "edge_type_cost": rb["edge_type_cost"],
        "bad_u_base": rb["u_base_tracklet_id"],
        "bad_v_base": rb["v_base_tracklet_id"],
        "alt_u_base": alt["u_base_tracklet_id"],
        "alt_v_base": alt["v_base_tracklet_id"],
        "bad_total_cost": rb["total_cost"],
        "alt_total_cost": alt["total_cost"],
    }
    row["delta_total_cost"] = row["bad_total_cost"] - row["alt_total_cost"]
    for col in [
        "term_env","term_time","term_vreq","term_group_coherence","term_flags",
        "term_birth_prior","term_death_prior","term_merge_prior","term_split_prior",
    ]:
        row[f"bad_{col}"] = rb.get(col)
        row[f"alt_{col}"] = alt.get(col)
        if pd.notna(rb.get(col)) and pd.notna(alt.get(col)):
            row[f"delta_{col}"] = rb.get(col) - alt.get(col)
        else:
            row[f"delta_{col}"] = pd.NA
    pair_rows.append(row)

pairwise_cmp = pd.DataFrame(pair_rows)
print("\nBad edges with a candidate good alternative:", pairwise_cmp.shape)
if not pairwise_cmp.empty:
    display(pairwise_cmp.sort_values("delta_total_cost")[[
        "bad_edge_id","alt_edge_id","edge_type_cost",
        "bad_u_base","bad_v_base","alt_u_base","alt_v_base",
        "bad_total_cost","alt_total_cost","delta_total_cost",
    ]])
    misranked = pairwise_cmp[pairwise_cmp["delta_total_cost"] < 0].copy()
    print("\nPairs where BAD edge is cheaper than ALT good edge (delta_total_cost < 0):", len(misranked))
    if not misranked.empty:
        delta_cols = [
            "delta_term_env","delta_term_time","delta_term_vreq","delta_term_group_coherence",
            "delta_term_flags","delta_term_birth_prior","delta_term_death_prior",
            "delta_term_merge_prior","delta_term_split_prior","delta_total_cost",
        ]
        print("\nMean deltas for mis-ranked pairs (BAD - ALT):")
        display(misranked[delta_cols].mean().to_frame("mean_delta"))
        ex = misranked.sort_values("delta_total_cost").iloc[0]
        print("\nExample mis-ranked pair: BAD edge then ALT edge explanation")
        bad_row = edges_with_gt[edges_with_gt["edge_id"] == ex["bad_edge_id"]].iloc[0]
        alt_row = edges_with_gt[edges_with_gt["edge_id"] == ex["alt_edge_id"]].iloc[0]
        print("\nBAD edge explanation:")
        explain_edge(bad_row, d2_cfg)
        print("\nALT (good) edge explanation:")
        explain_edge(alt_row, d2_cfg)

Bad selected edges: 7
Good selected edges: 17

Bad edges with a candidate good alternative: (6, 37)


,bad_edge_id,alt_edge_id,edge_type_cost,bad_u_base,bad_v_base,alt_u_base,alt_v_base,bad_total_cost,alt_total_cost,delta_total_cost
0,E:MERGE:T:t10->G:548-757:carrier=t4:d=t10:n=t13,E:MERGE:T:t1:s1:138-280->G:281-332:carrier=t4:...,EdgeType.MERGE,t10,t4,t1,t4,-0.390000,-0.156189,-0.233811
2,E:MERGE:T:t9->G:427-507:carrier=t4:d=t9:n=t10,E:MERGE:T:t1:s1:138-280->G:281-332:carrier=t4:...,EdgeType.MERGE,t9,t4,t1,t4,-0.365252,-0.156189,-0.209063
4,E:SPLIT:G:427-507:carrier=t4:d=t9:n=t10->T:t10,E:SPLIT:G:0-137:carrier=t1:d=none:n=t4->T:t4:s...,EdgeType.SPLIT,t4,t10,t1,t4,-0.388454,-0.250810,-0.137643
3,E:SPLIT:G:281-332:carrier=t4:d=t1:n=t9->T:t9,E:SPLIT:G:0-137:carrier=t1:d=none:n=t4->T:t4:s...,EdgeType.SPLIT,t4,t9,t1,t4,-0.383390,-0.250810,-0.132580
5,E:SPLIT:G:548-757:carrier=t4:d=t10:n=t13->T:t13,E:SPLIT:G:0-137:carrier=t1:d=none:n=t4->T:t4:s...,EdgeType.SPLIT,t4,t13,t1,t4,-0.377808,-0.250810,-0.126997
1,E:MERGE:T:t13->G:811-900:carrier=t4:d=t13:n=none,E:MERGE:T:t1:s1:138-280->G:281-332:carrier=t4:...,EdgeType.MERGE,t13,t4,t1,t4,-0.097898,-0.156189,0.058291



Pairs where BAD edge is cheaper than ALT good edge (delta_total_cost < 0): 5

Mean deltas for mis-ranked pairs (BAD - ALT):


,mean_delta
delta_term_env,0.000000
delta_term_time,0.000000
delta_term_vreq,-0.168019
delta_term_group_coherence,0.000000
delta_term_flags,0.000000
delta_term_birth_prior,0.000000
delta_term_death_prior,0.000000
delta_term_merge_prior,0.000000
delta_term_split_prior,0.000000
delta_total_cost,-0.168019



Example mis-ranked pair: BAD edge then ALT edge explanation

BAD edge explanation:
edge_id: E:MERGE:T:t10->G:548-757:carrier=t4:d=t10:n=t13
u: T:t10 (base_tracklet=t10, gt=E2)
v: G:548-757:carrier=t4:d=t10:n=t13 (base_tracklet=t4, gt=E1)
edge_type: EdgeType.MERGE  |  gt_class: bad  |  is_selected: True
total_cost: -0.39

Features:
  dist_m: 0.0
  dt_s: nan
  v_req_mps: nan
  contact_rel: nan

term_vreq (motion + MERGE/SPLIT distance):
  value: 0.0
  config: w_vreq=1.0, v_cost_scale_mps=None, v_hinge_mps=None
  intuition: higher required velocity / distance beyond thresholds should increase this term;
            currently bad edges often have lower term_vreq than good ones → vreq is relatively weak.

term_group_coherence (group behavior):
  value: -0.5
  config: bonus_group_coherent=0.5, penalty_group_incoherent=0.5
  raw group_coherent feature (if present): nan
  intuition: strongly negative values mean the solver is being *rewarded* for this edge on group grounds;
            our ag

## 8. Auto-tuning D2 Weights for cam03


Use the GT-aware analytics above to iteratively tweak a small set of D2 weights, rerun Stage D→D3 for this clip, and log how stitching quality changes.

In [29]:
# 8.1 Define tuning targets and candidate weight sets
import copy

# Path to default.yaml for config edits
CONFIG_DIR = PROJECT_ROOT / "configs"
DEFAULT_YAML_PATH = CONFIG_DIR / "default.yaml"
print("Using config:", DEFAULT_YAML_PATH)

# Load a pristine copy of the full default config
with DEFAULT_YAML_PATH.open("r") as f:
    default_cfg_template = yaml.safe_load(f)

baseline_d2_cfg = copy.deepcopy(default_cfg_template["stages"]["stage_D"]["d2_costs"])
print("Baseline D2 costs:")
display(baseline_d2_cfg)

# We will tune only the dominant drivers we observed:
# - w_vreq (term_vreq)
# - bonus_group_coherent / penalty_group_incoherent (term_group_coherence)
# - w_flags (term_flags)
TUNED_KEYS = [
    "w_vreq",
    "bonus_group_coherent",
    "penalty_group_incoherent",
    "w_flags",
]

# Build a small set of candidate configurations to explore
candidate_weight_sets = []

# Epoch 0: baseline (no changes)
candidate_weight_sets.append({
    "epoch": 0,
    "name": "baseline",
    "overrides": {k: baseline_d2_cfg.get(k) for k in TUNED_KEYS},
})

# Epoch 1: stronger vreq penalty
candidate_weight_sets.append({
    "epoch": 1,
    "name": "vreq_x1.5",
    "overrides": {
        "w_vreq": baseline_d2_cfg.get("w_vreq", 1.0) * 1.5,
        "bonus_group_coherent": baseline_d2_cfg.get("bonus_group_coherent"),
        "penalty_group_incoherent": baseline_d2_cfg.get("penalty_group_incoherent"),
        "w_flags": baseline_d2_cfg.get("w_flags"),
    },
})

# Epoch 2: weaker vreq penalty
candidate_weight_sets.append({
    "epoch": 2,
    "name": "vreq_x0.5",
    "overrides": {
        "w_vreq": baseline_d2_cfg.get("w_vreq", 1.0) * 0.5,
        "bonus_group_coherent": baseline_d2_cfg.get("bonus_group_coherent"),
        "penalty_group_incoherent": baseline_d2_cfg.get("penalty_group_incoherent"),
        "w_flags": baseline_d2_cfg.get("w_flags"),
    },
})

# Epoch 3: stronger group coherence reward/penalty
candidate_weight_sets.append({
    "epoch": 3,
    "name": "group_x1.5",
    "overrides": {
        "w_vreq": baseline_d2_cfg.get("w_vreq", 1.0),
        "bonus_group_coherent": baseline_d2_cfg.get("bonus_group_coherent", 0.5) * 1.5,
        "penalty_group_incoherent": baseline_d2_cfg.get("penalty_group_incoherent", 0.5) * 1.5,
        "w_flags": baseline_d2_cfg.get("w_flags"),
    },
})

print("\nCandidate weight sets:")
for c in candidate_weight_sets:
    print(c["epoch"], c["name"], c["overrides"])

Using config: /Users/bryanthomas/Desktop/Professional/Projects/roll_tracker/configs/default.yaml
Baseline D2 costs:


{'enabled': True,
 'missing_geom_policy': 'disallow',
 'dt_max_s': 1.0,
 'v_cost_scale_mps': None,
 'v_hinge_mps': None,
 'w_time': 0.1,
 'w_vreq': 1.0,
 'base_env_cost': 0.01,
 'use_flags': True,
 'w_flags': 0.25,
 'use_contact_rel': True,
 'contact_conf_floor': 0.25,
 'contact_rel_alpha': 0.35,
 'bonus_group_coherent': 0.75,
 'penalty_group_incoherent': 0.75,
 'birth_cost': 2.0,
 'death_cost': 2.0,
 'merge_prior': 0.1,
 'split_prior': 0.1,
 'reconnect_extra_env_cost': 1.0,
 'reconnect_w_z': 1.0,
 'reconnect_w_time': 0.0}


Candidate weight sets:
0 baseline {'w_vreq': 1.0, 'bonus_group_coherent': 0.75, 'penalty_group_incoherent': 0.75, 'w_flags': 0.25}
1 vreq_x1.5 {'w_vreq': 1.5, 'bonus_group_coherent': 0.75, 'penalty_group_incoherent': 0.75, 'w_flags': 0.25}
2 vreq_x0.5 {'w_vreq': 0.5, 'bonus_group_coherent': 0.75, 'penalty_group_incoherent': 0.75, 'w_flags': 0.25}
3 group_x1.5 {'w_vreq': 1.0, 'bonus_group_coherent': 1.125, 'penalty_group_incoherent': 1.125, 'w_flags': 0.25}


In [30]:
# 8.2 Helper to write per-epoch configs (snapshot + default.yaml)
def make_and_write_epoch_config(epoch: int, overrides: dict):
    cfg = copy.deepcopy(default_cfg_template)
    d2 = cfg["stages"]["stage_D"]["d2_costs"]
    for k, v in overrides.items():
        if k not in d2:
            print(f"[epoch {epoch}] WARNING: key {k} not in d2_costs, adding it.")
        d2[k] = v
    # Snapshot for this epoch
    epoch_path = CONFIG_DIR / f"default_epoch_{epoch}.yaml"
    with epoch_path.open("w") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    # Overwrite default.yaml so the pipeline picks up this epoch's weights
    with DEFAULT_YAML_PATH.open("w") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    print(f"[epoch {epoch}] Wrote {epoch_path} and updated {DEFAULT_YAML_PATH}.")

In [39]:
# 8.3 Helper to re-run Stage D→D3 for this cam03 clip
import subprocess

# Hard-code the project venv python so bjj_pipeline is importable
ORCH_PYTHON = str(PROJECT_ROOT / ".venv" / "bin" / "python")

# Hard-coded input path for this evaluation clip (matches CLIP_ID)
CLIP_INPUT_PATH = PROJECT_ROOT / "data" / "raw" / "nest" / "cam03" / "2026-01-03" / "12" / f"{CLIP_ID}.mp4"
print("Clip input path:", CLIP_INPUT_PATH)
print("Using orchestrator python:", ORCH_PYTHON)

def run_stage_D_for_cam03():
    """Run Stage D (through D3) for the configured cam03 clip using current default.yaml.

    This mirrors the CLI you run in the terminal:
      .venv/bin/python -m bjj_pipeline.stages.orchestration.cli run ...
    and executes from PROJECT_ROOT so the package is on sys.path.

    IMPORTANT: The current pipeline continues past D3 into a downstream
    Stage D/stitch step that expects person_tracks.parquet. For this
    notebook we only need D0–D3 artifacts, so we treat a non-zero
    return code as *non-fatal* if all required D artifacts exist.
    """
    cmd = [
        ORCH_PYTHON,
        "-m", "bjj_pipeline.stages.orchestration.cli",
        "run",
        "--clip", str(CLIP_INPUT_PATH),
        "--camera", CAMERA_ID,
        "--mode", "multiplex_AC",
        "--to-stage", "D",
        "--force-stage", "D",
    ]
    print("Running (cwd=PROJECT_ROOT):", " ".join(cmd))
    result = subprocess.run(cmd, cwd=str(PROJECT_ROOT), capture_output=True, text=True)
    print("Return code:", result.returncode)
    if result.stdout:
        print("--- stdout ---")
        print(result.stdout)
    if result.stderr:
        print("--- stderr ---")
        print(result.stderr)
    # Check that the artifacts we actually consume for evaluation exist
    required_paths = [
        D1_NODES_PATH,
        D1_EDGES_PATH,
        D2_COSTS_PATH,
        D3_COMPILED_COSTS_PATH,
        D3_COMPILED_EDGES_PATH,
        D3_SELECTED_EDGES_PATH,
    ]
    missing = [str(p) for p in required_paths if not p.exists()]
    if missing:
        raise RuntimeError(f"Stage D run did not produce required artifacts; missing: {missing}")
    if result.returncode != 0:
        print("[warn] Stage D CLI returned non-zero exit code, but D0–D3 artifacts are present; proceeding anyway for tuning.")

Clip input path: /Users/bryanthomas/Desktop/Professional/Projects/roll_tracker/data/raw/nest/cam03/2026-01-03/12/cam03-20260103-124000_0-30s.mp4
Using orchestrator python: /Users/bryanthomas/Desktop/Professional/Projects/roll_tracker/.venv/bin/python


In [32]:
# 8.4 Helpers to rebuild edges_with_gt and compute pairwise mis-ranking metrics
def rebuild_edges_with_gt_for_current_outputs() -> pd.DataFrame:
    """Reload Stage D artifacts for this clip and rebuild edges_with_gt from scratch."""
    # Reload D1 nodes and D3 compiled artifacts
    d1_nodes_local = pq.read_table(D1_NODES_PATH).to_pandas()
    d3_compiled_costs_local = pq.read_table(D3_COMPILED_COSTS_PATH).to_pandas()
    d3_compiled_edges_local = pq.read_table(D3_COMPILED_EDGES_PATH).to_pandas()
    d3_selected_local = pq.read_table(D3_SELECTED_EDGES_PATH).to_pandas()

    # Unified edge table with selection flag
    d3_compiled_edges_local["edge_id"] = d3_compiled_edges_local["edge_id"].astype(str)
    d3_compiled_costs_local["edge_id"] = d3_compiled_costs_local["edge_id"].astype(str)
    edges_with_costs_local = d3_compiled_edges_local.merge(
        d3_compiled_costs_local, on="edge_id", how="left", suffixes=("_edge", "_cost")
)
    sel_local = d3_selected_local[["edge_id", "flow"]].copy()
    sel_local["edge_id"] = sel_local["edge_id"].astype(str)
    edges_with_costs_local = edges_with_costs_local.merge(sel_local, on="edge_id", how="left")
    edges_with_costs_local["is_selected"] = edges_with_costs_local["flow"].fillna(0).astype(int) > 0

    # Attach base_tracklet_id metadata
    nodes_idx_local = d1_nodes_local.set_index("node_id")
    u_meta_local = nodes_idx_local[["base_tracklet_id", "segment_type"]].rename(
        columns={"base_tracklet_id": "u_base_tracklet_id", "segment_type": "u_segment_type"}
)
    v_meta_local = nodes_idx_local[["base_tracklet_id", "segment_type"]].rename(
        columns={"base_tracklet_id": "v_base_tracklet_id", "segment_type": "v_segment_type"}
)
    edges_enriched_local = edges_with_costs_local.merge(
        u_meta_local, left_on="u", right_index=True, how="left"
).merge(
        v_meta_local, left_on="v", right_index=True, how="left"
)

    # Attach GT labels using existing tracklet_to_gt mapping
    def _lookup_gt(tid: str) -> str | None:
        return tracklet_to_gt.get(tid)

    edges_enriched_local["u_gt"] = edges_enriched_local["u_base_tracklet_id"].map(_lookup_gt)
    edges_enriched_local["v_gt"] = edges_enriched_local["v_base_tracklet_id"].map(_lookup_gt)

    def _classify_edge(row: pd.Series) -> str:
        u_gt = row["u_gt"]
        v_gt = row["v_gt"]
        if pd.isna(u_gt) or pd.isna(v_gt):
            return "unknown"
        return "good" if u_gt == v_gt else "bad"

    edges_enriched_local["gt_class"] = edges_enriched_local.apply(_classify_edge, axis=1)
    return edges_enriched_local


def compute_pairwise_misranking(edges_with_gt_local: pd.DataFrame) -> dict:
    """Compute pairwise BAD vs best-good ranking stats for current edges_with_gt."""
    bad_sel_local = edges_with_gt_local[(edges_with_gt_local["is_selected"]) & (edges_with_gt_local["gt_class"] == "bad")].copy()
    good_sel_local = edges_with_gt_local[(edges_with_gt_local["is_selected"]) & (edges_with_gt_local["gt_class"] == "good")].copy()

    def _find_best_alt_edge(row_bad: pd.Series, good_edges: pd.DataFrame) -> pd.Series | None:
        candidates = good_edges
        edge_type = row_bad.get("edge_type_cost")
        if edge_type is not None:
            candidates = candidates[candidates["edge_type_cost"] == edge_type]
        if candidates.empty:
            return None
        u_bt = row_bad.get("u_base_tracklet_id")
        v_bt = row_bad.get("v_base_tracklet_id")
        mask = pd.Series(False, index=candidates.index)
        if pd.notna(u_bt):
            mask |= (candidates["u_base_tracklet_id"] == u_bt) | (candidates["v_base_tracklet_id"] == u_bt)
        if pd.notna(v_bt):
            mask |= (candidates["u_base_tracklet_id"] == v_bt) | (candidates["v_base_tracklet_id"] == v_bt)
        candidates = candidates[mask]
        if candidates.empty:
            return None
        return candidates.sort_values("total_cost").iloc[0]

    pair_rows_local = []
    for _, rb in bad_sel_local.iterrows():
        alt = _find_best_alt_edge(rb, good_sel_local)
        if alt is None:
            continue
        row = {
            "bad_edge_id": rb["edge_id"],
            "alt_edge_id": alt["edge_id"],
            "edge_type_cost": rb["edge_type_cost"],
            "bad_total_cost": rb["total_cost"],
            "alt_total_cost": alt["total_cost"],
        }
        row["delta_total_cost"] = row["bad_total_cost"] - row["alt_total_cost"]
        pair_rows_local.append(row)

    if not pair_rows_local:
        return {
            "n_misranked_pairs": 0,
            "mean_delta_total_cost_misranked": float("nan"),
        }

    pairwise_cmp_local = pd.DataFrame(pair_rows_local)
    misranked_local = pairwise_cmp_local[pairwise_cmp_local["delta_total_cost"] < 0].copy()
    n_mis = int(len(misranked_local))
    if n_mis == 0:
        mean_delta = float("nan")
    else:
        mean_delta = float(misranked_local["delta_total_cost"].mean())
    return {
        "n_misranked_pairs": n_mis,
        "mean_delta_total_cost_misranked": mean_delta,
    }

In [33]:
# 8.5 Evaluate a single config (one epoch)
def evaluate_current_config(epoch: int, name: str, overrides: dict) -> dict:
    print(f"\n===== EPOCH {epoch} ({name}) =====")
    print("Overrides:", overrides)
    # Run Stage D with current default.yaml (already updated by make_and_write_epoch_config)
    run_stage_D_for_cam03()
    # Rebuild edges_with_gt from fresh artifacts
    edges_epoch = rebuild_edges_with_gt_for_current_outputs()
    # Basic counts
    sel_epoch = edges_epoch[edges_epoch["is_selected"]].copy()
    n_sel = int(len(sel_epoch))
    n_bad = int(len(sel_epoch[sel_epoch["gt_class"] == "bad"]))
    n_good = int(len(sel_epoch[sel_epoch["gt_class"] == "good"]))
    print(f"Selected edges: total={n_sel}, good={n_good}, bad={n_bad}")
    # Pairwise mis-ranking stats
    pair_stats = compute_pairwise_misranking(sel_epoch)
    print("Pairwise mis-ranking stats:", pair_stats)
    result = {
        "epoch": epoch,
        "name": name,
        "n_selected": n_sel,
        "n_good_selected": n_good,
        "n_bad_selected": n_bad,
    }
    result.update(pair_stats)
    # Also log the tuned weights explicitly
    for k, v in overrides.items():
        result[f"w_{k}"] = v
    return result

In [41]:
# 8.6 Main tuning loop over candidate weight sets
tuning_results = []
for cand in candidate_weight_sets:
    epoch = cand["epoch"]
    name = cand["name"]
    overrides = cand["overrides"]
    make_and_write_epoch_config(epoch, overrides)
    try:
        res = evaluate_current_config(epoch, name, overrides)
        tuning_results.append(res)
    except Exception as e:
        print(f"[epoch {epoch}] ERROR during evaluation:", e)

if tuning_results:
    tuning_df = pd.DataFrame(tuning_results)
    print("\nTuning results (sorted by n_bad_selected, then n_misranked_pairs):")
    display(tuning_df.sort_values(["n_bad_selected", "n_misranked_pairs"]))
else:
    print("No successful epochs recorded.")

[epoch 0] Wrote /Users/bryanthomas/Desktop/Professional/Projects/roll_tracker/configs/default_epoch_0.yaml and updated /Users/bryanthomas/Desktop/Professional/Projects/roll_tracker/configs/default.yaml.

===== EPOCH 0 (baseline) =====
Overrides: {'w_vreq': 1.0, 'bonus_group_coherent': 0.75, 'penalty_group_incoherent': 0.75, 'w_flags': 0.25}
Running (cwd=PROJECT_ROOT): /Users/bryanthomas/Desktop/Professional/Projects/roll_tracker/.venv/bin/python -m bjj_pipeline.stages.orchestration.cli run --clip /Users/bryanthomas/Desktop/Professional/Projects/roll_tracker/data/raw/nest/cam03/2026-01-03/12/cam03-20260103-124000_0-30s.mp4 --camera cam03 --mode multiplex_AC --to-stage D --force-stage D
Return code: 2
--- stdout ---
[D1 DEBUG] nodes_df columns and dtypes before write:
 node_id                          object
node_type                        object
capacity                          int64
start_frame                       Int64
end_frame                         Int64
base_tracklet_id      

,epoch,name,n_selected,n_good_selected,n_bad_selected,n_misranked_pairs,mean_delta_total_cost_misranked,w_w_vreq,w_bonus_group_coherent,w_penalty_group_incoherent,w_w_flags
3,3,group_x1.5,31,15,7,3,-0.156343,1.0,1.125,1.125,0.25
0,0,baseline,31,14,8,3,-0.156343,1.0,0.750,0.750,0.25
1,1,vreq_x1.5,31,14,8,3,-0.234514,1.5,0.750,0.750,0.25
2,2,vreq_x0.5,32,13,10,2,-0.058804,0.5,0.750,0.750,0.25
